In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import glob
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

# ============================================================
# 2. Dataset path
# ============================================================

DATA_PATH = "./GEFCom2014-L_V2/Load"


tasks = sorted(
    glob.glob(DATA_PATH + "/Task */L*-train.csv")
)

print("Number of tasks:", len(tasks))

Number of tasks: 15


In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_PATH = Path("./GEFCom2014-L_V2/Load")
weather_cols = [f"w{i}" for i in range(1, 26)]


def task_number(path):
    match = re.fullmatch(r"L(\d+)-train", path.stem)
    if match is None:
        raise ValueError(f"Unexpected filename: {path.name}")
    return int(match.group(1))


def parse_task_timestamps(raw, number):
    """
    For the supplied GEFCom2014 archive.

    Raw dates have no separators or zero padding, so individual
    strings can be ambiguous. Reconstruct each task's hourly
    sequence and validate EVERY timestamp against the raw string.
    """
    if number == 1:
        start = pd.Timestamp("2001-01-01 01:00:00")
    else:
        start = (
            pd.Timestamp("2010-10-01 01:00:00")
            + pd.DateOffset(months=number - 2)
        )

    timestamps = pd.date_range(
        start=start,
        periods=len(raw),
        freq="h",
    )

    expected = [
        f"{t.month}{t.day}{t.year} {t.hour}:00"
        for t in timestamps
    ]
    actual = raw.astype(str).str.strip().tolist()

    for row, (observed, required) in enumerate(zip(actual, expected)):
        if observed != required:
            raise ValueError(
                f"Task {number}, row {row}: "
                f"raw timestamp {observed!r} does not match "
                f"expected {required!r}. Check the source file."
            )

    return timestamps


tasks = sorted(
    DATA_PATH.glob("Task */L*-train.csv"),
    key=task_number,
)

if [task_number(p) for p in tasks] != list(range(1, 16)):
    raise ValueError(
        "Expected training files for Tasks 1–15. Check DATA_PATH."
    )

all_data = []

for file in tasks:
    temp = pd.read_csv(file)
    temp["TIMESTAMP"] = parse_task_timestamps(
        temp["TIMESTAMP"],
        task_number(file),
    )
    all_data.append(temp)

df = (
    pd.concat(all_data, ignore_index=True)
    .sort_values(["ZONEID", "TIMESTAMP"])
    .reset_index(drop=True)
)

if df.duplicated(["ZONEID", "TIMESTAMP"]).any():
    raise ValueError("Unexpected duplicate zone/timestamp pairs.")

if df["ZONEID"].nunique() != 1:
    raise ValueError("This pipeline expects one zone.")

# The supplied archive has an initial weather-only period.
# Remove it; no resampling or interpolation is needed.
df_hourly = (
    df.loc[
        df["LOAD"].notna(),
        ["TIMESTAMP", "LOAD"] + weather_cols,
    ]
    .reset_index(drop=True)
)

assert len(df_hourly) == 60600
assert df_hourly[["LOAD"] + weather_cols].notna().all().all()

hourly_steps = df_hourly["TIMESTAMP"].diff().iloc[1:]
assert hourly_steps.eq(pd.Timedelta(hours=1)).all()

print("Raw rows:", len(df))
print("Labelled rows:", len(df_hourly))
print("Start:", df_hourly["TIMESTAMP"].iloc[0])
print("End:", df_hourly["TIMESTAMP"].iloc[-1])
print("Non-hourly gaps:", (~hourly_steps.eq(pd.Timedelta(hours=1))).sum())

Raw rows: 95664
Labelled rows: 60600
Start: 2005-01-01 01:00:00
End: 2011-12-01 00:00:00
Non-hourly gaps: 0


In [3]:
class EnergyDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        df,
        history=168,
        horizon=24
    ):

        self.history = history
        self.horizon = horizon


        # Past demand
        self.demand = (
            df["LOAD_scaled"]
            .values
            .astype(np.float32)
        )


        # Weather
        weather_cols = [
            f"w{i}_scaled"
            for i in range(1,26)
        ]


        self.weather = (
            df[weather_cols]
            .values
            .astype(np.float32)
        )


        # Calendar
        calendar_cols = [
            "hour_sin",
            "hour_cos",
            "dow_sin",
            "dow_cos",
            "month_sin",
            "month_cos"
        ]


        self.calendar = (
            df[calendar_cols]
            .values
            .astype(np.float32)
        )


    def __len__(self):
        return max(
            0,
            len(self.demand) - self.history - self.horizon + 1
        )


    def __getitem__(self, idx):

        start = idx
        end = idx + self.history

        target_start = end
        target_end = end + self.horizon


        # past inputs

        past_demand = self.demand[
            start:end
        ]

        past_weather = self.weather[
            start:end
        ]

        past_calendar = self.calendar[
            start:end
        ]


        # future known variables

        future_weather = self.weather[
            target_start:target_end
        ]

        future_calendar = self.calendar[
            target_start:target_end
        ]


        # previous day load pattern
        # last 24 hours before forecast

        last_day = self.demand[
            end-24:end
        ]


        # target

        target = self.demand[
            target_start:target_end
        ]


        return {

            "past_demand": torch.tensor(
                past_demand,
                dtype=torch.float32
            ).unsqueeze(-1),


            "past_weather": torch.tensor(
                past_weather,
                dtype=torch.float32
            ),


            "past_calendar": torch.tensor(
                past_calendar,
                dtype=torch.float32
            ),


            "future_weather": torch.tensor(
                future_weather,
                dtype=torch.float32
            ),


            "future_calendar": torch.tensor(
                future_calendar,
                dtype=torch.float32
            ),


            "last_day": torch.tensor(
                last_day,
                dtype=torch.float32
            ),


            "target": torch.tensor(
                target,
                dtype=torch.float32
            )

        }

In [4]:
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

HISTORY = 168
HORIZON = 24

df_hourly = (
    df_hourly
    .sort_values("TIMESTAMP")
    .reset_index(drop=True)
    .copy()
)

t = pd.to_datetime(df_hourly["TIMESTAMP"])
df_hourly["TIMESTAMP"] = t

assert t.diff().iloc[1:].eq(pd.Timedelta(hours=1)).all()

# Create calendar features before making split copies.
for name, values, period in [
    ("hour", t.dt.hour, 24),
    ("dow", t.dt.dayofweek, 7),
    ("month", t.dt.month, 12),
]:
    df_hourly[f"{name}_sin"] = np.sin(
        2 * np.pi * values / period
    )
    df_hourly[f"{name}_cos"] = np.cos(
        2 * np.pi * values / period
    )

# 70% training / 7.5% validation / 7.5% calibration / 15% test
n = len(df_hourly)

train_end = int(0.70 * n)
val_end = int(0.775 * n)
cal_end = int(0.85 * n)

split_spec = {
    "n": n,
    "train_end": train_end,
    "val_end": val_end,
    "cal_end": cal_end,
    "first": str(t.iloc[0]),
    "last": str(t.iloc[-1]),
}

# Include history context; targets remain within each block.
train_df = df_hourly.iloc[:train_end].copy()
val_df = df_hourly.iloc[train_end - HISTORY:val_end].copy()
cal_df = df_hourly.iloc[val_end - HISTORY:cal_end].copy()
test_df = df_hourly.iloc[cal_end - HISTORY:].copy()

weather_cols = [f"w{i}" for i in range(1, 26)]
scaled_weather_cols = [f"{c}_scaled" for c in weather_cols]

load_scaler = StandardScaler().fit(train_df[["LOAD"]])
weather_scaler = StandardScaler().fit(train_df[weather_cols])

for frame in [train_df, val_df, cal_df, test_df]:
    frame["LOAD_scaled"] = load_scaler.transform(
        frame[["LOAD"]]
    ).ravel()

    frame[scaled_weather_cols] = weather_scaler.transform(
        frame[weather_cols]
    )

    assert np.isfinite(
        frame[["LOAD_scaled"] + scaled_weather_cols]
    ).all().all()

train_dataset = EnergyDataset(
    train_df, history=HISTORY, horizon=HORIZON
)
val_dataset = EnergyDataset(
    val_df, history=HISTORY, horizon=HORIZON
)
cal_dataset = EnergyDataset(
    cal_df, history=HISTORY, horizon=HORIZON
)
test_dataset = EnergyDataset(
    test_df, history=HISTORY, horizon=HORIZON
)

for name, frame, dataset in [
    ("Train", train_df, train_dataset),
    ("Validation", val_df, val_dataset),
    ("Calibration", cal_df, cal_dataset),
    ("Test", test_df, test_dataset),
]:
    assert len(dataset) == len(frame) - HISTORY - HORIZON + 1

    print(
        name,
        "| Windows:", len(dataset),
        "| First target:", frame["TIMESTAMP"].iloc[HISTORY],
    )

train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=64, shuffle=False, num_workers=0
)
cal_loader = DataLoader(
    cal_dataset, batch_size=64, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False, num_workers=0
)

Train | Windows: 42229 | First target: 2005-01-08 01:00:00
Validation | Windows: 4522 | First target: 2009-11-03 13:00:00
Calibration | Windows: 4522 | First target: 2010-05-11 22:00:00
Test | Windows: 9067 | First target: 2010-11-17 07:00:00


In [5]:
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


class OrderedQuantileHead(nn.Module):
    """
    Input:  [batch, horizon, features]
    Output: [batch, horizon, 99]

    Quantile levels: 0.01, ..., 0.99
    Median index: 49
    """

    def __init__(self, input_dim):
        super().__init__()

        self.hidden = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.GELU(),
        )
        self.output = nn.Linear(128, 99)

        nn.init.normal_(self.output.weight, std=0.005)

        with torch.no_grad():
            self.output.bias.zero_()

            initial_spacing = 0.02
            inverse_softplus = math.log(
                math.expm1(initial_spacing)
            )
            self.output.bias[1:].fill_(inverse_softplus)

    def forward(self, x):
        raw = self.output(self.hidden(x))

        median = raw[..., :1]

        lower_steps = F.softplus(raw[..., 1:50])
        upper_steps = F.softplus(raw[..., 50:99])

        lower = median - torch.cumsum(
            lower_steps, dim=-1
        ).flip(-1)

        upper = median + torch.cumsum(
            upper_steps, dim=-1
        )

        return torch.cat(
            [lower, median, upper],
            dim=-1,
        )


class CausalConv1d(nn.Module):
    """Left padding ensures each output uses only earlier inputs."""

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
    ):
        super().__init__()

        self.left_padding = (
            kernel_size - 1
        ) * dilation

        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0,
        )

    def forward(self, x):
        x = F.pad(x, (self.left_padding, 0))
        return self.conv(x)


class TemporalBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
        dropout,
    ):
        super().__init__()

        self.net = nn.Sequential(
            CausalConv1d(
                in_channels,
                out_channels,
                kernel_size,
                dilation,
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            CausalConv1d(
                out_channels,
                out_channels,
                kernel_size,
                dilation,
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.residual = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1,
            )
        )

    def forward(self, x):
        return F.relu(
            self.net(x) + self.residual(x)
        )


class BaselineForecast(nn.Module):
    """
    Shared probabilistic forecasting interface.

    Historical inputs:
        demand:   [B, 168, 1]
        weather:  [B, 168, 25]
        calendar: [B, 168, 6]

    Future inputs:
        weather:  [B, 24, 25]
        calendar: [B, 24, 6]

    Output:
        ordered quantiles [B, 24, 99]
    """

    def __init__(
        self,
        architecture,
        d_model=128,
        horizon=24,
        history=168,
        dropout=0.2,
    ):
        super().__init__()

        allowed = {
            "gru",
            "lstm",
            "tcn",
            "patch_transformer",
        }

        if architecture not in allowed:
            raise ValueError(
                f"architecture must be one of {sorted(allowed)}"
            )

        if horizon != 24:
            raise ValueError(
                "The previous-day residual requires horizon=24."
            )

        self.architecture = architecture
        self.history = history
        self.horizon = horizon

        self.register_buffer(
            "taus",
            torch.arange(1, 100).float() / 100,
        )

        input_dim = 1 + 25 + 6

        if architecture == "gru":
            self.encoder = nn.GRU(
                input_size=input_dim,
                hidden_size=d_model,
                num_layers=2,
                batch_first=True,
                dropout=dropout,
            )

        elif architecture == "lstm":
            self.encoder = nn.LSTM(
                input_size=input_dim,
                hidden_size=d_model,
                num_layers=2,
                batch_first=True,
                dropout=dropout,
            )

        elif architecture == "tcn":
            kernel_size = 3
            dilations = [1, 2, 4, 8, 16, 32]

            blocks = []
            in_channels = input_dim

            for dilation in dilations:
                blocks.append(
                    TemporalBlock(
                        in_channels=in_channels,
                        out_channels=d_model,
                        kernel_size=kernel_size,
                        dilation=dilation,
                        dropout=dropout,
                    )
                )
                in_channels = d_model

            self.encoder = nn.Sequential(*blocks)

            self.receptive_field = (
                1
                + 2
                * (kernel_size - 1)
                * sum(dilations)
            )

            if self.receptive_field < history:
                raise ValueError(
                    "TCN receptive field does not cover history."
                )

        else:
            self.patch_len = 24
            self.patch_stride = 12

            if history < self.patch_len:
                raise ValueError(
                    "History must be at least one patch long."
                )

            if (
                history - self.patch_len
            ) % self.patch_stride != 0:
                raise ValueError(
                    "Patch settings must cover the full history."
                )

            num_patches = (
                (history - self.patch_len)
                // self.patch_stride
                + 1
            )

            self.patch_projection = nn.Linear(
                input_dim * self.patch_len,
                d_model,
            )

            self.patch_position = nn.Parameter(
                torch.zeros(
                    1,
                    num_patches,
                    d_model,
                )
            )
            nn.init.normal_(
                self.patch_position,
                std=0.02,
            )

            # Construct layers separately for independent initialization.
            self.encoder = nn.ModuleList([
                nn.TransformerEncoderLayer(
                    d_model=d_model,
                    nhead=4,
                    dim_feedforward=256,
                    dropout=dropout,
                    activation="gelu",
                    batch_first=True,
                )
                for _ in range(3)
            ])

            self.patch_summary = nn.Sequential(
                nn.Flatten(start_dim=1),
                nn.Linear(
                    num_patches * d_model,
                    d_model,
                ),
                nn.GELU(),
                nn.Dropout(dropout),
            )

        # Shared decoder: all architectures use future covariates.
        self.future_projection = nn.Sequential(
            nn.Linear(25 + 6, d_model),
            nn.GELU(),
        )

        self.horizon_embedding = nn.Parameter(
            torch.zeros(1, horizon, d_model)
        )
        nn.init.normal_(
            self.horizon_embedding,
            std=0.02,
        )

        self.decoder = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
        )

        self.head = OrderedQuantileHead(d_model)

    def encode_history(self, x):
        if self.architecture == "gru":
            _, hidden = self.encoder(x)
            return hidden[-1]

        if self.architecture == "lstm":
            _, (hidden, _) = self.encoder(x)
            return hidden[-1]

        if self.architecture == "tcn":
            features = self.encoder(
                x.transpose(1, 2)
            )
            return features[:, :, -1]

        # unfold returns [B, patches, channels, patch_len].
        patches = x.unfold(
            dimension=1,
            size=self.patch_len,
            step=self.patch_stride,
        )

        patches = patches.contiguous().flatten(
            start_dim=2
        )

        tokens = (
            self.patch_projection(patches)
            + self.patch_position
        )

        for layer in self.encoder:
            tokens = layer(tokens)

        return self.patch_summary(tokens)

    def forward(
        self,
        past_demand,
        past_weather,
        past_calendar,
        future_weather,
        future_calendar,
        last_day,
    ):
        history = torch.cat(
            [
                past_demand,
                past_weather,
                past_calendar,
            ],
            dim=-1,
        )

        if history.size(1) != self.history:
            raise ValueError(
                f"Expected {self.history} historical steps."
            )

        context = self.encode_history(history)

        context = context.unsqueeze(1).expand(
            -1,
            self.horizon,
            -1,
        )

        future = torch.cat(
            [
                future_weather,
                future_calendar,
            ],
            dim=-1,
        )

        future = (
            self.future_projection(future)
            + self.horizon_embedding
        )

        features = self.decoder(
            torch.cat(
                [context, future],
                dim=-1,
            )
        )

        return (
            last_day.unsqueeze(-1)
            + self.head(features)
        )


def pinball_loss(quantiles, target, taus):
    error = target.unsqueeze(-1) - quantiles

    return torch.maximum(
        taus * error,
        (taus - 1) * error,
    ).mean()


INPUT_KEYS = (
    "past_demand",
    "past_weather",
    "past_calendar",
    "future_weather",
    "future_calendar",
    "last_day",
)


def batch_inputs(batch, device):
    return [
        batch[key].to(device)
        for key in INPUT_KEYS
    ]

In [7]:
import random
import time
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

ARCHITECTURE = "patch_transformer"
SEED = 42
EPOCHS = 30
PATIENCE = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

config = {
    "architecture": ARCHITECTURE,
    "d_model": 128,
    "horizon": HORIZON,
    "history": HISTORY,
    "dropout": 0.2,
}

model = BaselineForecast(**config).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

CHECKPOINT_DIR = Path("checkpoints") / ARCHITECTURE
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT = (
    CHECKPOINT_DIR
    / f"{ARCHITECTURE}_quantile_seed{SEED}_best.pt"
).resolve()

CALIBRATED_CHECKPOINT = (
    CHECKPOINT_DIR
    / f"{ARCHITECTURE}_quantile_seed{SEED}_calibrated.pt"
).resolve()

print("Device:", device)
print("Architecture:", ARCHITECTURE)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad),
)
print("Raw checkpoint:", CHECKPOINT)
print("Calibrated checkpoint:", CALIBRATED_CHECKPOINT)

if ARCHITECTURE == "tcn":
    print("TCN receptive field:", model.receptive_field)


def run_epoch(loader, training):
    model.train(training)

    total_loss = 0.0
    total_samples = 0

    progress = tqdm(
        loader,
        desc="Train" if training else "Validation",
        leave=False,
    )

    with torch.set_grad_enabled(training):
        for batch in progress:
            target = batch["target"].to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            quantiles = model(*batch_inputs(batch, device))
            loss = pinball_loss(
                quantiles,
                target,
                model.taus,
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError("Non-finite pinball loss.")

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )
                optimizer.step()

            batch_size = target.size(0)
            total_loss += loss.detach().item() * batch_size
            total_samples += batch_size

            progress.set_postfix(
                pinball=f"{total_loss / total_samples:.6f}"
            )

    if total_samples == 0:
        raise ValueError("Empty DataLoader.")

    return total_loss / total_samples


best_val = float("inf")
bad_epochs = 0
saved_this_run = False

history = {
    "train_pinball": [],
    "val_pinball": [],
}

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_score = run_epoch(train_loader, training=True)
    val_score = run_epoch(val_loader, training=False)

    scheduler.step(val_score)

    history["train_pinball"].append(train_score)
    history["val_pinball"].append(val_score)

    improved = val_score < best_val

    if improved:
        best_val = val_score
        bad_epochs = 0

        state = {
            "version": "v5.1-raw",
            "epoch": epoch,
            "config": config,
            "seed": SEED,
            "split_spec": split_spec,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_pinball": val_score,
            "load_mean": load_scaler.mean_.tolist(),
            "load_scale": load_scaler.scale_.tolist(),
            "weather_mean": weather_scaler.mean_.tolist(),
            "weather_scale": weather_scaler.scale_.tolist(),
        }

        temporary = CHECKPOINT.with_suffix(".tmp")
        torch.save(state, temporary)
        temporary.replace(CHECKPOINT)
        saved_this_run = True

    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train: {train_score:.6f} | "
        f"Val: {val_score:.6f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
        f"Time: {time.time() - start:.1f}s"
        + (" | SAVED" if improved else "")
    )

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

if not saved_this_run:
    raise RuntimeError("No checkpoint was saved during this run.")

print("Best validation pinball:", best_val)

Device: mps
Architecture: patch_transformer
Trainable parameters: 780259
Raw checkpoint: /Users/pratyush/Coding/Projects/dreamformer/checkpoints/patch_transformer/patch_transformer_quantile_seed42_best.pt
Calibrated checkpoint: /Users/pratyush/Coding/Projects/dreamformer/checkpoints/patch_transformer/patch_transformer_quantile_seed42_calibrated.pt


Epoch 01/30 | Train: 0.094861 | Val: 0.090355 | LR: 1.00e-04 | Time: 65.7s | SAVED


Epoch 02/30 | Train: 0.074610 | Val: 0.082469 | LR: 1.00e-04 | Time: 55.0s | SAVED


Epoch 03/30 | Train: 0.068051 | Val: 0.081259 | LR: 1.00e-04 | Time: 53.7s | SAVED


Epoch 04/30 | Train: 0.063010 | Val: 0.073194 | LR: 1.00e-04 | Time: 67.7s | SAVED


Epoch 05/30 | Train: 0.059293 | Val: 0.072312 | LR: 1.00e-04 | Time: 61.2s | SAVED


... (18 outputs left out)

Epoch 24/30 | Train: 0.034989 | Val: 0.064230 | LR: 5.00e-05 | Time: 33.3s


Epoch 25/30 | Train: 0.034665 | Val: 0.065167 | LR: 5.00e-05 | Time: 33.5s


Epoch 26/30 | Train: 0.034395 | Val: 0.063506 | LR: 5.00e-05 | Time: 33.6s


Epoch 27/30 | Train: 0.034169 | Val: 0.063412 | LR: 2.50e-05 | Time: 33.5s
Early stopping.
Best validation pinball: 0.06212504719512118


In [11]:
from pathlib import Path
import torch

expected_path = Path(CHECKPOINT).resolve()

print("Current CHECKPOINT:", globals().get("CHECKPOINT"))
print("Expected checkpoint:", expected_path)
print("File exists:", expected_path.is_file())

if not expected_path.is_file():
    raise FileNotFoundError(expected_path)

saved = torch.load(
    expected_path,
    map_location="cpu",
    weights_only=True,
)

print("Saved object type:", type(saved).__name__)

if not isinstance(saved, dict):
    raise TypeError("Expected a checkpoint dictionary.")

print("Available keys:", list(saved.keys()))
print("Version:", saved.get("version", "<missing>"))
print("Epoch:", saved.get("epoch", "<missing>"))
print("Validation pinball:", saved.get("val_pinball", "<missing>"))
print("Split specification:", saved.get("split_spec", "<missing>"))

required = {
    "version",
    "split_spec",
    "model_state_dict",
    "load_mean",
    "load_scale",
    "weather_mean",
    "weather_scale",
}

missing = required.difference(saved)

if missing:
    print("\nMissing required fields:", sorted(missing))
    print("Keep this file unchanged until its origin is identified.")
else:
    assert saved["version"] == "v5.1-raw"
    assert saved["split_spec"] == split_spec

    CHECKPOINT = expected_path
    print("\nCheckpoint metadata matches. Rerun the calibration cell.")

Current CHECKPOINT: /Users/pratyush/Coding/Projects/dreamformer/checkpoints/patch_transformer/patch_transformer_quantile_seed42_best.pt
Expected checkpoint: /Users/pratyush/Coding/Projects/dreamformer/checkpoints/patch_transformer/patch_transformer_quantile_seed42_best.pt
File exists: True
Saved object type: dict
Available keys: ['version', 'epoch', 'config', 'seed', 'split_spec', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'val_pinball', 'load_mean', 'load_scale', 'weather_mean', 'weather_scale']
Version: v5.1-raw
Epoch: 19
Validation pinball: 0.06212504719512118
Split specification: {'n': 60600, 'train_end': 42420, 'val_end': 46965, 'cal_end': 51510, 'first': '2005-01-01 01:00:00', 'last': '2011-12-01 00:00:00'}

Checkpoint metadata matches. Rerun the calibration cell.


In [12]:
checkpoint = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=True,
)

assert checkpoint["config"] == config, (
    "Checkpoint configuration does not match this model."
)
assert checkpoint["version"] == "v5.1-raw"
assert checkpoint["split_spec"] == split_spec

for scaler, prefix in [
    (load_scaler, "load"),
    (weather_scaler, "weather"),
]:
    assert np.allclose(
        scaler.mean_,
        checkpoint[f"{prefix}_mean"],
    )
    assert np.allclose(
        scaler.scale_,
        checkpoint[f"{prefix}_scale"],
    )

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)
model.eval()

print("Loaded epoch:", checkpoint["epoch"])


@torch.no_grad()
def predict_raw(loader):
    model.eval()

    predictions = []
    targets = []

    for batch in loader:
        q = model(*batch_inputs(batch, device))

        predictions.append(q.cpu().numpy())
        targets.append(batch["target"].cpu().numpy())

    q = np.concatenate(predictions, axis=0)
    y = np.concatenate(targets, axis=0)

    scale = float(load_scaler.scale_[0])
    mean = float(load_scaler.mean_[0])

    return q * scale + mean, y * scale + mean


def fit_side_scale(offsets, residual, levels):
    """
    Minimize average pinball loss over a single positive scale.

    offsets: [samples, 49], quantile minus median
    residual: [samples], actual minus median
    levels: [49], corresponding quantile levels

    The objective is convex and piecewise linear.
    Its minimizer is a weighted quantile of residual/offset.
    """
    d = np.asarray(offsets, dtype=np.float64)

    r = np.broadcast_to(
        np.asarray(residual, dtype=np.float64)[:, None],
        d.shape,
    )

    tau = np.broadcast_to(
        np.asarray(levels, dtype=np.float64),
        d.shape,
    )

    mask = d != 0

    if not mask.any():
        return 1.0

    weight = np.abs(d[mask])
    ratio = r[mask] / d[mask]

    alpha = np.where(
        d[mask] > 0,
        tau[mask],
        1 - tau[mask],
    )

    order = np.argsort(ratio)
    cumulative = np.cumsum(weight[order])
    target_weight = np.sum(weight * alpha)

    index = min(
        np.searchsorted(cumulative, target_weight),
        len(order) - 1,
    )

    return max(float(ratio[order[index]]), 1e-6)


def apply_calibration(q, lower_scale, upper_scale):
    result = q.copy()
    median = q[..., 49:50]

    result[..., :49] = (
        median
        + lower_scale[None, :, None]
        * (q[..., :49] - median)
    )

    result[..., 50:] = (
        median
        + upper_scale[None, :, None]
        * (q[..., 50:] - median)
    )

    # Median index 49 is never modified.
    assert np.isfinite(result).all()
    assert np.array_equal(result[..., 49], q[..., 49])
    assert (np.diff(result, axis=-1) >= -1e-5).all()

    return result


# Calibration data only: no test predictions or targets used here.
cal_q, cal_y = predict_raw(cal_loader)

assert np.isfinite(cal_q).all()
assert np.isfinite(cal_y).all()

taus = np.arange(1, 100) / 100.0

lower_scale = np.ones(HORIZON)
upper_scale = np.ones(HORIZON)

for h in range(HORIZON):
    median = cal_q[:, h, 49].astype(np.float64)
    residual = cal_y[:, h].astype(np.float64) - median

    lower_scale[h] = fit_side_scale(
        cal_q[:, h, :49].astype(np.float64) - median[:, None],
        residual,
        taus[:49],
    )

    upper_scale[h] = fit_side_scale(
        cal_q[:, h, 50:].astype(np.float64) - median[:, None],
        residual,
        taus[50:],
    )

cal_q_adjusted = apply_calibration(
    cal_q,
    lower_scale,
    upper_scale,
)

# Save model weights, preprocessing metadata, and calibration together.
bundle = dict(checkpoint)
bundle["version"] = "v5.1-calibrated"
bundle["lower_scale"] = lower_scale.tolist()
bundle["upper_scale"] = upper_scale.tolist()

temporary = CALIBRATED_CHECKPOINT.with_suffix(".tmp")
torch.save(bundle, temporary)
temporary.replace(CALIBRATED_CHECKPOINT)

print("Saved calibrated checkpoint:", CALIBRATED_CHECKPOINT)

print(
    pd.DataFrame({
        "horizon": np.arange(1, HORIZON + 1),
        "lower_scale": lower_scale,
        "upper_scale": upper_scale,
    })
    .round(3)
    .to_string(index=False)
)

Loaded epoch: 19
Saved calibrated checkpoint: /Users/pratyush/Coding/Projects/dreamformer/checkpoints/patch_transformer/patch_transformer_quantile_seed42_calibrated.pt
 horizon  lower_scale  upper_scale
       1        0.819        1.449
       2        0.822        1.543
       3        0.844        1.666
       4        0.868        1.800
       5        0.911        1.873
       6        0.962        1.947
       7        0.979        1.991
       8        0.989        2.040
       9        1.016        2.060
      10        1.029        2.101
      11        1.062        2.085
      12        1.095        2.067
      13        1.151        2.048
      14        1.163        2.051
      15        1.217        2.006
      16        1.212        2.017
      17        1.234        1.984
      18        1.234        1.989
      19        1.217        1.965
      20        1.188        1.929
      21        1.176        1.875
      22        1.142        1.799
      23        1.114      

In [13]:
bundle = torch.load(
    CALIBRATED_CHECKPOINT,
    map_location=device,
    weights_only=True,
)

assert bundle["version"] == "v5.1-calibrated"
assert bundle["config"] == config
assert bundle["split_spec"] == split_spec

for scaler, prefix in [
    (load_scaler, "load"),
    (weather_scaler, "weather"),
]:
    assert np.allclose(
        scaler.mean_,
        bundle[f"{prefix}_mean"],
    )
    assert np.allclose(
        scaler.scale_,
        bundle[f"{prefix}_scale"],
    )

model.load_state_dict(
    bundle["model_state_dict"],
    strict=True,
)
model.eval()

lower_scale = np.asarray(bundle["lower_scale"])
upper_scale = np.asarray(bundle["upper_scale"])

raw_quantile_pred, y_true = predict_raw(test_loader)

quantile_pred = apply_calibration(
    raw_quantile_pred,
    lower_scale,
    upper_scale,
)

pred = quantile_pred[..., 49]
taus = np.arange(1, 100) / 100.0


def mean_pinball_numpy(q, y):
    total = 0.0
    levels = np.arange(1, 100) / 100.0

    for start in range(0, len(y), 256):
        error = (
            y[start:start + 256, :, None]
            - q[start:start + 256]
        )

        total += np.maximum(
            levels * error,
            (levels - 1) * error,
        ).sum(dtype=np.float64)

    return total / q.size


def report(q, y, label):
    error = q[..., 49] - y
    valid = np.abs(y) > 1e-8

    mape = (
        100 * np.abs(error[valid] / y[valid]).mean()
        if valid.any()
        else float("nan")
    )

    print("\n" + "=" * 65)
    print(label)
    print(f"Median MAE: {np.abs(error).mean():.4f}")
    print(f"Median RMSE: {np.sqrt((error ** 2).mean()):.4f}")
    print(f"Median MAPE: {mape:.4f}%")
    print(f"Mean pinball: {mean_pinball_numpy(q, y):.4f}")

    rows = []

    for lo, hi in [(10, 90), (5, 95), (1, 99)]:
        lower = q[..., lo - 1]
        upper = q[..., hi - 1]

        rows.append({
            "interval": f"{hi - lo}%",
            "coverage_pct": 100 * (
                (y >= lower) & (y <= upper)
            ).mean(),
            "below_lower_pct": 100 * (y < lower).mean(),
            "above_upper_pct": 100 * (y > upper).mean(),
            "mean_width": (upper - lower).mean(),
        })

    print(pd.DataFrame(rows).round(3).to_string(index=False))


report(
    raw_quantile_pred,
    y_true,
    f"TEST: RAW {ARCHITECTURE.upper()}",
)

report(
    quantile_pred,
    y_true,
    f"TEST: CALIBRATED {ARCHITECTURE.upper()}",
)

assert np.array_equal(
    pred,
    raw_quantile_pred[..., 49],
)

print("\nMedian forecasts are unchanged by calibration.")

report(cal_q, cal_y, "CALIBRATION BLOCK: RAW")
report(cal_q_adjusted, cal_y, "CALIBRATION BLOCK: ADJUSTED")


TEST: RAW PATCH_TRANSFORMER
Median MAE: 7.8040
Median RMSE: 12.4779
Median MAPE: 6.3674%
Mean pinball: 2.9355
interval  coverage_pct  below_lower_pct  above_upper_pct  mean_width
     80%        64.096           21.742           14.162   15.783000
     90%        75.831           14.607            9.562   20.721001
     98%        89.294            6.355            4.350   30.260000

TEST: CALIBRATED PATCH_TRANSFORMER
Median MAE: 7.8040
Median RMSE: 12.4779
Median MAPE: 6.3674%
Mean pinball: 2.9386
interval  coverage_pct  below_lower_pct  above_upper_pct  mean_width
     80%        75.404           20.113            4.483   23.153000
     90%        84.819           13.097            2.083   30.437000
     98%        94.102            5.451            0.447   44.610001

Median forecasts are unchanged by calibration.

CALIBRATION BLOCK: RAW
Median MAE: 6.9400
Median RMSE: 9.5900
Median MAPE: 4.2208%
Mean pinball: 2.5880
interval  coverage_pct  below_lower_pct  above_upper_pct  mean_wid